# load_memory — research notebook

First node of the quiz agent: reads what we know about the learner and writes a `LearnerContext` into state, which `plan_quiz` uses to personalize the blueprint. Runs as a **node** (always executes) but is also registered as a `@tool` so a future agent can query memory on demand.

**Resilience rule: tools never raise.** Memory is an enhancement, not a dependency — any failure degrades to an empty context and the quiz still generates.

Prototyped on `InMemoryStore`; `PostgresStore` has the same interface.

In [ ]:
# pip install -q langgraph langchain-core   (in research/requirements.txt)
from typing import TypedDict
from langchain_core.tools import tool
from langgraph.store.memory import InMemoryStore

class MasteryRecord(TypedDict):
    score: float      # 0..1, exponential moving average
    seen: int
    correct: int

class QuizHistoryEntry(TypedDict):
    url: str
    domain: str
    question_hashes: list[str]

class LearnerContext(TypedDict):
    mastered_concepts: list[str]       # score >= 0.8 -> skip or harder Bloom level
    weak_concepts: list[str]           # score <= 0.4 -> prioritize
    recent_question_hashes: list[str]  # avoid repeats

class GraphState(TypedDict, total=False):
    page_url: str
    learner_context: LearnerContext

store = InMemoryStore()

## Memory layout

- `("users", user_id, "mastery")` — key: concept, value: `MasteryRecord`
- `("users", user_id, "quiz_history")` — key: quiz id, value: `QuizHistoryEntry`

History is scoped by page domain, a cheap topic proxy for V1.

In [ ]:
import hashlib
USER = "user_nject"

def qhash(q: str) -> str:
    return hashlib.sha256(q.lower().strip().encode()).hexdigest()[:12]

mastery: dict[str, MasteryRecord] = {
    "self-attention":       {"score": 0.9,  "seen": 5, "correct": 5},
    "multi-head attention": {"score": 0.35, "seen": 4, "correct": 1},
    "layer normalization":  {"score": 0.2,  "seen": 2, "correct": 0},
}
for concept, rec in mastery.items():
    store.put(("users", USER, "mastery"), concept, rec)

history: QuizHistoryEntry = {
    "url": "https://en.wikipedia.org/wiki/Transformer_(deep_learning)",
    "domain": "en.wikipedia.org",
    "question_hashes": [qhash("What does self-attention compute?")],
}
store.put(("users", USER, "quiz_history"), "quiz_abc123", history)

## The node (and tool)

In [ ]:
from urllib.parse import urlparse

MASTERED_AT, WEAK_AT, MIN_SEEN = 0.8, 0.4, 2

def empty_context() -> LearnerContext:
    return LearnerContext(mastered_concepts=[], weak_concepts=[], recent_question_hashes=[])

@tool
def load_memory(page_url: str, user_id: str) -> LearnerContext:
    """Load the learner's mastery profile and recent question history for this page's domain.
    Never raises: any failure degrades to an empty context."""
    try:
        domain = urlparse(page_url).netloc
        mastered, weak, hashes = [], [], []

        for item in store.search(("users", user_id, "mastery")):
            rec: MasteryRecord = item.value
            if rec["seen"] < MIN_SEEN:
                continue
            if rec["score"] >= MASTERED_AT:
                mastered.append(item.key)
            elif rec["score"] <= WEAK_AT:
                weak.append(item.key)

        for item in store.search(("users", user_id, "quiz_history")):
            entry: QuizHistoryEntry = item.value
            if entry["domain"] == domain:
                hashes.extend(entry["question_hashes"])

        return LearnerContext(mastered_concepts=mastered, weak_concepts=weak,
                              recent_question_hashes=hashes[-100:])
    except Exception:                            # noqa: BLE001 — resilience boundary
        return empty_context()

def load_memory_node(state: GraphState, *, user_id: str = USER) -> GraphState:
    """Graph node: thin wrapper, same logic serves both roles."""
    return {"learner_context": load_memory.func(state["page_url"], user_id)}

state: GraphState = {"page_url": "https://en.wikipedia.org/wiki/Attention_Is_All_You_Need"}
load_memory_node(state)

In [ ]:
# cold start AND failure both degrade to the same empty context, never a crash
print(load_memory_node({"page_url": "https://example.com/x"}, user_id="brand_new_user"))
print(load_memory_node({"page_url": 12345}))   # bad input -> empty context

## Closing the loop: update_mastery

Runs on `POST /attempts`, not in the graph. EMA so recent performance dominates.

In [ ]:
ALPHA = 0.3

def update_mastery(user_id: str, concept: str, correct: bool) -> MasteryRecord:
    ns = ("users", user_id, "mastery")
    item = store.get(ns, concept)
    rec: MasteryRecord = item.value if item else {"score": 0.5, "seen": 0, "correct": 0}
    rec["score"] = round((1 - ALPHA) * rec["score"] + ALPHA * correct, 3)
    rec["seen"] += 1
    rec["correct"] += int(correct)
    store.put(ns, concept, rec)
    return rec

print(update_mastery(USER, "layer normalization", True))   # 0.2 -> 0.44
print(update_mastery(USER, "layer normalization", True))   # 0.44 -> 0.61
print(load_memory_node(state)["learner_context"]["weak_concepts"])  # left the weak list

## Notes

- **Never raises.** Store down or bad input returns an empty context; log the exception in production.
- Node vs tool: the node always runs; `@tool` is for future agent-initiated lookups (fact-check flow).
- `user_id` comes from API auth via `configurable`, never from page data.
- Swap `InMemoryStore()` for `PostgresStore` in production; node code unchanged.
- Open question: hash question text (exact repeats) or (concept, bloom_level) pairs (near-duplicates)?